# Unidad 2: Aprendizaje Automático 
## 🚢 DT vs LR con Validación Cruzada
### Inteligencia Artificial - Lic. en Sistemas - FCAD/UNER


![Individualidad](https://raw.githubusercontent.com/CristianPacifico/ia-ls-fcad-uner/main/notebooks/ml/images/pexels-jan-van-der-wolf-11680885-9771484.jpg)
[Foto de Jan van der Wolf](https://www.pexels.com/es-es/foto/rojo-blanco-sillas-asientos-9771484/)

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/ia-ls-fcad-uner/blob/main/notebooks/ml/11_DTvsLR_Titanic_CrossValidation.ipynb)

## Dataset: Titanic — Evaluación robusta con K-Fold Cross-Validation

En este notebook extendemos la comparación entre **Decision Tree** y **Logistic Regression** con una técnica más confiable de evaluación:

### ❓ ¿Por qué usar Validación Cruzada?

Con un único `train_test_split`, la métrica obtenida depende de **cómo se particionaron los datos** (el `random_state`). Esto puede darnos una evaluación optimista o pesimista del modelo.

La **K-Fold Cross-Validation** divide el dataset en *K* partes iguales (folds), entrena el modelo K veces usando K-1 folds y evalúa en el fold restante. Luego promedia los resultados:

$$\text{CV Score} = \frac{1}{K} \sum_{i=1}^{K} \text{Score}_i$$

Esto reduce la varianza de la estimación del rendimiento real del modelo. 🎯

## 📦 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn import metrics

## 📂 2. Carga y preparación del dataset

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/CristianPacifico/ia-ls-fcad-uner/main/data/ml/titanic.csv')
df['male'] = df['Sex'] == 'male'

print("🔍 Primeras filas del dataset:")
print(df.head())
print(f"\n📐 Estructura del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")

## ✂️ 3. Definición de features y split inicial

Primero hacemos un split tradicional para evaluar el rendimiento base, y luego comparamos con la validación cruzada.

In [ ]:
feature_cols = ['Pclass', 'male', 'Age', 'Siblings/Spouses', 'Parents/Children', 'Fare']
X = df[feature_cols].values
y = df['Survived'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=5
)

print(f"🏋️  Entrenamiento: {X_train.shape[0]} muestras")
print(f"🧪  Testeo:        {X_test.shape[0]} muestras")

## 🏗️ 4. Entrenamiento y evaluación simple (sin CV)

In [ ]:
modelDT = DecisionTreeClassifier()
modelLR = LogisticRegression()

modelDT.fit(X_train, y_train)
modelLR.fit(X_train, y_train)

y_predDT = modelDT.predict(X_test)
y_predLR = modelLR.predict(X_test)

acc_DT_simple = modelDT.score(X_test, y_test)
acc_LR_simple = modelLR.score(X_test, y_test)

print(f"🌳 DT  Accuracy (split simple): {acc_DT_simple:.4f}")
print(f"📈 LR  Accuracy (split simple): {acc_LR_simple:.4f}")

## 🔁 5. Evaluación con K-Fold Cross-Validation (K=5)

`cross_val_score` entrena y evalúa el modelo automáticamente en cada uno de los K folds. Devuelve un array con el score de cada fold.

```
Dataset completo:
[ Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 ]
  [test]    train    train    train    train   → Score 1
   train   [test]    train    train    train   → Score 2
   train    train   [test]    train    train   → Score 3
   train    train    train   [test]    train   → Score 4
   train    train    train    train   [test]   → Score 5
```

In [ ]:
K = 5
kf = KFold(n_splits=K, shuffle=True, random_state=42)

# Validación cruzada con accuracy
scores_DT = cross_val_score(DecisionTreeClassifier(), X, y, cv=kf, scoring='accuracy')
scores_LR = cross_val_score(LogisticRegression(), X, y, cv=kf, scoring='accuracy')

print(f"🌳 Decision Tree  — Scores por fold: {scores_DT.round(4)}")
print(f"   Media: {scores_DT.mean():.4f}  |  Std: {scores_DT.std():.4f}")
print()
print(f"📈 Logistic Reg.  — Scores por fold: {scores_LR.round(4)}")
print(f"   Media: {scores_LR.mean():.4f}  |  Std: {scores_LR.std():.4f}")

## 📊 6. Comparación: Split simple vs Cross-Validation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Gráfico 1: Score por fold ---
folds = [f'Fold {i+1}' for i in range(K)]
axes[0].plot(folds, scores_DT, marker='o', color='steelblue', label='Decision Tree')
axes[0].plot(folds, scores_LR, marker='s', color='darkorange', label='Logistic Reg.')
axes[0].axhline(scores_DT.mean(), color='steelblue', linestyle='--', alpha=0.6)
axes[0].axhline(scores_LR.mean(), color='darkorange', linestyle='--', alpha=0.6)
axes[0].set_title('Accuracy por fold (K=5)')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0.6, 1.0)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Gráfico 2: Comparación simple vs CV ---
labels = ['DT\nSplit simple', 'DT\nCV (media)', 'LR\nSplit simple', 'LR\nCV (media)']
values = [acc_DT_simple, scores_DT.mean(), acc_LR_simple, scores_LR.mean()]
colors = ['steelblue', 'cornflowerblue', 'darkorange', 'sandybrown']
bars = axes[1].bar(labels, values, color=colors)
axes[1].set_ylim(0.6, 1.0)
axes[1].set_title('Comparación Split simple vs Cross-Validation')
axes[1].set_ylabel('Accuracy')
axes[1].bar_label(bars, fmt='%.4f', padding=3)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('DT vs LR — Titanic | Evaluación con K-Fold CV', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 📏 7. Múltiples métricas con Cross-Validation

In [ ]:
from sklearn.model_selection import cross_validate

scoring_metrics = ['accuracy', 'precision', 'recall', 'f1']

cv_results_DT = cross_validate(DecisionTreeClassifier(), X, y, cv=kf, scoring=scoring_metrics)
cv_results_LR = cross_validate(LogisticRegression(), X, y, cv=kf, scoring=scoring_metrics)

print("📊 Resultados con 5-Fold Cross-Validation (media ± std)\n")
print(f"{'Métrica':<12} {'🌳 Decision Tree':>22} {'📈 Logistic Reg.':>22}")
print("-" * 58)
for m in scoring_metrics:
    key = f'test_{m}'
    dt_mean, dt_std = cv_results_DT[key].mean(), cv_results_DT[key].std()
    lr_mean, lr_std = cv_results_LR[key].mean(), cv_results_LR[key].std()
    print(f"{m:<12} {dt_mean:.4f} ± {dt_std:.4f}       {lr_mean:.4f} ± {lr_std:.4f}")

## 🏁 8. Conclusiones

- 🔁 La **validación cruzada** proporciona una estimación más honesta y estable del rendimiento comparado con un único split.
- 📉 Un modelo con **alta varianza entre folds** puede estar sobreajustándose.
- 🌳 El DT sin restricciones tiende a tener más varianza entre folds (inestabilidad).
- 📈 La LR suele ser más estable y tiene menor desviación estándar entre folds.
- 💡 Usa `cross_validate` en lugar de `cross_val_score` cuando necesites múltiples métricas al mismo tiempo.